# Project 2: Retail Demand & Basket Analysis
Dataset: [Online Retail II UCI](https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci)

**Project Overview:** 
This end-to-end data science project analyzes historical B2B/wholesale e-commerce transactions (2009–2011) to solve three core retail challenges:
1. **Demand Forecasting:** Predicting daily and weekly unit demand using time-series feature engineering, machine learning regressors, and SARIMAX.
2. **Market Basket Analysis:** Discovering high-affinity product combinations using the FP-Growth algorithm to drive cross-selling and bundling strategies.
3. **Commercial Risk Management:** Identifying high-return products and high-value churn-risk customer segments through Return Rate Matrices and RFM analysis.

## Load dataset

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/online_retail_II.csv")
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## Checking dataset


In [4]:
# Missing values checking
def check_missing_values(df):
  missing_values = df.isna().sum()
  missing_values = missing_values[missing_values > 0]
  missing_percentage = missing_values / len(df) * 100
  missing_percentage = missing_percentage[missing_percentage > 0].round(2).astype(str) + ' %'
  missing_display = pd.DataFrame({
      	'Missing Values': missing_values,
				'Missing Percentage': missing_percentage
	})
  return missing_display
  
# Duplicate values checking
def check_duplicate_values(df):
	duplicate_values = int(df.duplicated().sum())
	return duplicate_values

In [5]:
# 1. Dataset shape
print(f"Dataset shape: {df.shape}\n")

# 2. Dataset info
print("Dataset info:")
df.info()

# 3. Dataset description
print("\nDataset description:")
df.describe()

Dataset shape: (1067371, 8)

Dataset info:
<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  str    
 1   StockCode    1067371 non-null  str    
 2   Description  1062989 non-null  str    
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  str    
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 65.1 MB

Dataset description:


,Quantity,Price,Customer ID
count,1.067371e+06,1.067371e+06,824364.000000
mean,9.938898e+00,4.649388e+00,15324.638504
std,1.727058e+02,1.235531e+02,1697.464450
min,-8.099500e+04,-5.359436e+04,12346.000000
25%,1.000000e+00,1.250000e+00,13975.000000
50%,3.000000e+00,2.100000e+00,15255.000000
75%,1.000000e+01,4.150000e+00,16797.000000
max,8.099500e+04,3.897000e+04,18287.000000


In [6]:
# 4. Missing values
print("Missing values:")
check_missing_values(df) if check_missing_values(df).shape[0] > 0 else "No missing values found"

Missing values:


,Missing Values,Missing Percentage
Description,4382,0.41 %
Customer ID,243007,22.77 %


In [7]:
# 5. Duplicate values
print("\nDuplicate values:")
check_duplicate_values(df) if check_duplicate_values(df) > 0 else "No duplicate values found"


Duplicate values:


34335

- There are some missing values in `Description` column (with **0.41%** missing values) and `Customer ID` column (with **22.77%** missing values)
- There are 34,355 duplicated vales in this dataset

## Handling dataset

In [8]:
# Delete (space) in 'Customer ID' with replace(' ','')
df.columns = (
	df.columns
 .str.replace(' ','')
)

### 1. Handling duplicate values

In [9]:
# Delete the duplicated values in dataset
df = df.drop_duplicates().copy()

### 2. Data Imputation

#### 1. `Description` column

In [10]:
# Impute empty “Description” fields using a valid StockCode
valid_desc = df[df['Description'].notna()].drop_duplicates('StockCode').set_index('StockCode')['Description']
df['Description'] = np.where(df['Description'].isna(), df['StockCode'].map(valid_desc), df['Description'])
df = df.dropna(subset=['Description'])

#### 2. `CustomerID` column

In [11]:
# Mark empty "CustomerID" as a 'Guest' 
df['CustomerID'] = df['CustomerID'].fillna('Guest').astype(str).str.replace('.0','',regex=False)

C:\Users\Valenttanda\AppData\Local\Temp\ipykernel_21304\3985490472.py:2: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  df['CustomerID'] = df['CustomerID'].fillna('Guest').astype(str).str.replace('.0','',regex=False)


## Feature Engineering

In [12]:
df['StockCode'] = df['StockCode'].str.strip()
df['Description'] = df['Description'].str.strip()
df['Invoice'] = df['Invoice'].str.strip()
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [13]:
# Remove non-product code
non_product_code = ['POST', 'D', 'DOT', 'M', 'BANK CHARGES', 'AMAZONFEE', 'CRUK', 'C2']
df = df[~df['StockCode'].isin(non_product_code)]

In [14]:
# Split Returns vs Net Sales Transaction
returns_df = df[df['Invoice'].str.startswith('C') | (df['Quantity'] < 0)].copy()

clean_df = df[
	(~df['Invoice'].str.startswith('C')) &
	(df['Quantity'] > 0) &
  (df['Price'] > 0)
].copy()

# Add TotalRevenue column for future
clean_df['TotalRevenue'] = clean_df['Quantity'] * clean_df['Price']
print(f'Total Clean Sales Rows: {len(clean_df):,}')
print(f'Total Returns Rows: {len(returns_df):,}')

Total Clean Sales Rows: 1,003,492
Total Returns Rows: 21,139


### 1. Daily Feature Engineering

In [15]:
clean_df['Date'] = clean_df['InvoiceDate'].dt.floor('D')

# Daily aggregation
daily = clean_df.groupby('Date').agg(
	total_quantity=('Quantity','sum'),
	total_revenue=('TotalRevenue','sum'),
	unique_invoices=('Invoice','nunique')
)
daily

,total_quantity,total_revenue,unique_invoices
Date,,,
2009-12-01,26070,52438.61,117
2009-12-02,31799,61873.10,115
2009-12-03,49219,73059.78,124
2009-12-04,21192,39349.21,88
2009-12-05,5119,9803.05,30
...,...,...,...
2011-12-05,43674,80011.23,126
2011-12-06,30346,55257.31,114
2011-12-07,41766,72799.09,105


In [16]:
# Get the full date range from daily.index
date_range = pd.date_range(start=daily.index.min(), end=daily.index.max(), freq='D')
date_range

DatetimeIndex(['2009-12-01', '2009-12-02', '2009-12-03', '2009-12-04',
               '2009-12-05', '2009-12-06', '2009-12-07', '2009-12-08',
               '2009-12-09', '2009-12-10',
               ...
               '2011-11-30', '2011-12-01', '2011-12-02', '2011-12-03',
               '2011-12-04', '2011-12-05', '2011-12-06', '2011-12-07',
               '2011-12-08', '2011-12-09'],
              dtype='datetime64[us]', length=739, freq='D')

There is a difference in data length between the `daily` dataset and the `date_range`. This indicates that there are empty calendar entries in the `daily` dataset, so those empty entries need to be filled with a value of 0.

In [17]:
daily = daily.reindex(date_range, fill_value=0).reset_index().rename(columns={'index':'Date'})
daily

,Date,total_quantity,total_revenue,unique_invoices
0,2009-12-01,26070,52438.61,117
1,2009-12-02,31799,61873.10,115
2,2009-12-03,49219,73059.78,124
3,2009-12-04,21192,39349.21,88
4,2009-12-05,5119,9803.05,30
...,...,...,...,...
734,2011-12-05,43674,80011.23,126
735,2011-12-06,30346,55257.31,114
736,2011-12-07,41766,72799.09,105
737,2011-12-08,34991,77571.19,120


In [18]:
daily['day_of_week'] = daily['Date'].dt.dayofweek
daily['is_saturday'] = (daily['day_of_week'] == 5).astype(int)
daily['month'] = daily['Date'].dt.month
daily['quarter'] = daily['Date'].dt.quarter
daily['week_of_year'] = daily['Date'].dt.isocalendar().week.astype(int)

# Lag & Rolling
for lag in [1,7,14,28]:
  daily[f'qty_lag_{lag}'] = daily['total_quantity'].shift(lag)

daily['qty_roll_mean_7'] = daily['total_quantity'].shift(1).rolling(window=7).mean()
daily['qty_roll_std_7'] = daily['total_quantity'].shift(1).rolling(window=7).std()
daily['qty_roll_mean_28'] = daily['total_quantity'].shift(1).rolling(window=28).mean()
daily['invoices_lag_7'] = daily['unique_invoices'].shift(7)
daily

C:\Users\Valenttanda\AppData\Local\Temp\ipykernel_21304\1417575050.py:1: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  daily['day_of_week'] = daily['Date'].dt.dayofweek


,Date,total_quantity,total_revenue,unique_invoices,day_of_week,is_saturday,month,quarter,week_of_year,qty_lag_1,qty_lag_7,qty_lag_14,qty_lag_28,qty_roll_mean_7,qty_roll_std_7,qty_roll_mean_28,invoices_lag_7
0,2009-12-01,26070,52438.61,117,1,0,12,4,49,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2009-12-02,31799,61873.10,115,2,0,12,4,49,26070.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2009-12-03,49219,73059.78,124,3,0,12,4,49,31799.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2009-12-04,21192,39349.21,88,4,0,12,4,49,49219.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2009-12-05,5119,9803.05,30,5,1,12,4,49,21192.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
734,2011-12-05,43674,80011.23,126,0,0,12,4,49,12283.0,29311.0,25686.0,31282.0,22335.714286,11672.553730,24222.714286,114.0
735,2011-12-06,30346,55257.31,114,1,0,12,4,49,43674.0,30782.0,33654.0,26319.0,24387.571429,14110.800035,24665.285714,135.0
736,2011-12-07,41766,72799.09,105,2,0,12,4,49,30346.0,28324.0,39280.0,36927.0,24325.285714,14078.796489,24809.107143,107.0
737,2011-12-08,34991,77571.19,120,3,0,12,4,49,41766.0,26979.0,26365.0,38058.0,26245.571429,15554.481571,24981.928571,121.0


In [19]:
# Remove the missing values, then it's ready for regression
daily_model_df = daily.dropna().reset_index(drop=True)
daily_model_df

,Date,total_quantity,total_revenue,unique_invoices,day_of_week,is_saturday,month,quarter,week_of_year,qty_lag_1,qty_lag_7,qty_lag_14,qty_lag_28,qty_roll_mean_7,qty_roll_std_7,qty_roll_mean_28,invoices_lag_7
0,2009-12-29,0,0.00,0,1,0,12,4,53,0.0,14453.0,27058.0,26070.0,2718.571429,5448.298262,15187.571429,45.0
1,2009-12-30,0,0.00,0,2,0,12,4,53,0.0,4577.0,24881.0,31799.0,653.857143,1729.943393,14256.500000,22.0
2,2009-12-31,0,0.00,0,3,0,12,4,53,0.0,0.0,15403.0,49219.0,0.000000,0.000000,13120.821429,0.0
3,2010-01-01,0,0.00,0,4,0,1,1,53,0.0,0.0,17398.0,21192.0,0.000000,0.000000,11363.000000,0.0
4,2010-01-02,0,0.00,0,5,1,1,1,53,0.0,0.0,0.0,5119.0,0.000000,0.000000,10606.142857,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
706,2011-12-05,43674,80011.23,126,0,0,12,4,49,12283.0,29311.0,25686.0,31282.0,22335.714286,11672.553730,24222.714286,114.0
707,2011-12-06,30346,55257.31,114,1,0,12,4,49,43674.0,30782.0,33654.0,26319.0,24387.571429,14110.800035,24665.285714,135.0
708,2011-12-07,41766,72799.09,105,2,0,12,4,49,30346.0,28324.0,39280.0,36927.0,24325.285714,14078.796489,24809.107143,107.0
709,2011-12-08,34991,77571.19,120,3,0,12,4,49,41766.0,26979.0,26365.0,38058.0,26245.571429,15554.481571,24981.928571,121.0


### Weekly Feature Engineering

In [20]:
# Weekly aggregation
weekly = (daily.set_index('Date')
  .resample('W-SUN')
  .agg(
		total_quantity=('total_quantity','sum'),
		total_revenue=('total_revenue','sum'),
		unique_invoices=('unique_invoices','sum')
	)
  .reset_index())

weekly['week_of_year'] = weekly['Date'].dt.isocalendar().week.astype(int)
weekly['month'] = weekly['Date'].dt.month
weekly['quarter'] = weekly['Date'].dt.quarter
weekly['is_q4'] = (weekly['quarter'] == 4).astype(int)

# Lag & Rolling
for lag in [1,2,4]:
  weekly[f'qty_lag_{lag}'] = weekly['total_quantity'].shift(lag)

weekly['qty_roll_mean_4'] = weekly['total_quantity'].shift(1).rolling(window=4).mean()
weekly['qty_roll_std_4'] = weekly['total_quantity'].shift(1).rolling(window=4).std()
weekly['invoices_lag_1'] = weekly['unique_invoices'].shift(1)
weekly

,Date,total_quantity,total_revenue,unique_invoices,week_of_year,month,quarter,is_q4,qty_lag_1,qty_lag_2,qty_lag_4,qty_roll_mean_4,qty_roll_std_4,invoices_lag_1
0,2009-12-06,144924,260808.77,548,49,12,4,1,NaN,NaN,NaN,NaN,NaN,NaN
1,2009-12-13,116251,234033.21,525,50,12,4,1,144924.0,NaN,NaN,NaN,NaN,548.0
2,2009-12-20,136285,251674.56,491,51,12,4,1,116251.0,144924.0,NaN,NaN,NaN,525.0
3,2009-12-27,27792,51715.49,107,52,12,4,1,136285.0,116251.0,NaN,NaN,NaN,491.0
4,2010-01-03,0,0.00,0,53,1,1,0,27792.0,136285.0,144924.0,106313.00,53707.359489,107.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101,2011-11-13,182194,357706.72,634,45,11,4,1,158184.0,149054.0,117536.0,144260.25,18213.039621,527.0
102,2011-11-20,184889,373178.36,687,46,11,4,1,182194.0,158184.0,152267.0,160424.75,14997.378068,634.0
103,2011-11-27,154803,302392.22,616,47,11,4,1,184889.0,182194.0,149054.0,168580.25,17707.493411,687.0
104,2011-12-04,156350,309381.73,663,48,12,4,1,154803.0,184889.0,158184.0,170017.50,15715.612163,616.0


In [21]:
# Remove the missing values, then it's ready for regression
weekly_model_df = weekly.dropna().reset_index(drop=True)
weekly_model_df

,Date,total_quantity,total_revenue,unique_invoices,week_of_year,month,quarter,is_q4,qty_lag_1,qty_lag_2,qty_lag_4,qty_roll_mean_4,qty_roll_std_4,invoices_lag_1
0,2010-01-03,0,0.000,0,53,1,1,0,27792.0,136285.0,144924.0,106313.00,53707.359489,107.0
1,2010-01-10,96064,166055.790,202,1,1,1,0,0.0,27792.0,116251.0,70082.00,66368.528169,0.0
2,2010-01-17,76118,155491.820,263,2,1,1,0,96064.0,0.0,136285.0,65035.25,62332.318035,202.0
3,2010-01-24,135360,145957.901,263,3,1,1,0,76118.0,96064.0,27792.0,49993.50,43959.212592,263.0
4,2010-01-31,82931,153847.921,361,4,1,1,0,135360.0,76118.0,0.0,76885.50,56859.625919,263.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97,2011-11-13,182194,357706.720,634,45,11,4,1,158184.0,149054.0,117536.0,144260.25,18213.039621,527.0
98,2011-11-20,184889,373178.360,687,46,11,4,1,182194.0,158184.0,152267.0,160424.75,14997.378068,634.0
99,2011-11-27,154803,302392.220,616,47,11,4,1,184889.0,182194.0,149054.0,168580.25,17707.493411,687.0
100,2011-12-04,156350,309381.730,663,48,12,4,1,154803.0,184889.0,158184.0,170017.50,15715.612163,616.0


## EDA: Visualization

In [22]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ==========================================
# CHART 1: DAILY DEMAND & ROLLING MEAN TREND
# ==========================================
daily['roll_14'] = daily['total_quantity'].rolling(14).mean()

fig1 = go.Figure()
fig1.add_trace(go.Scatter(
	x=daily['Date'], y=daily['total_quantity'],
	mode='lines', name='Daily Quantity',
	line=dict(color='rgba(46, 134, 171, 0.35)', width=1)
))
fig1.add_trace(go.Scatter(
	x=daily['Date'], y=daily['roll_14'],
	mode='lines', name='14-Day Rolling Mean',
	line=dict(color='#E63946', width=2)
))
fig1.update_layout(
	title='EDA 1: Daily Demand Trends & Seasonal Patterns (2009-2011)',
	xaxis_title='Date',yaxis_title='Quantity', hovermode='x unified'
)
fig1.show()

In [23]:
# ==========================================
# CHART 2: DAILY DEMAND DISTRIBUTION OVER THE COURSE OF A WEEK
# ==========================================
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily['DayName'] = daily['Date'].dt.day_name()

fig2 = px.box(
    daily, x='DayName', y='total_quantity',
    category_orders={'DayName': day_order},
    color='DayName',
    title='EDA 2: Daily Demand Distribution (Saturday = 0)'
)
fig2.update_layout(showlegend=False, xaxis_title='Day', yaxis_title='Units Sold')
fig2.show()

In [24]:
# ==========================================
# CHART 3: AUTOCORRELATION (ACF) FOR LAG SELECTION
# ==========================================
lags = list(range(1,31))
acf_vals = [daily['total_quantity'].autocorr(lag=l) for l in lags]

fig3 = px.bar(
	x=lags, y=acf_vals,
	labels={'x': 'Lag (Past few Days)', 'y':'Autocorrelation (ACF)'},
	title='EDA 3: Demand Autocorrelation (Peaking at a Lag of 7)'
)
fig3.add_hline(y=0,line_color='black',line_width=1)
fig3.update_traces(marker_color=['#E63946' if (l % 7 == 0) else '#457B9D' for l in lags])
fig3.show()

In [25]:
# ==========================================
# CHART 4: BASKET SIZE DISTRIBUTION (ITEMS PER INVOICE)
# ==========================================
basket_sizes = clean_df.groupby('Invoice')['StockCode'].nunique().reset_index(name='UniqueItems')
fig4 = px.histogram(
	basket_sizes[basket_sizes['UniqueItems'] <= 50 ], # Limit to 50 items
	x='UniqueItems', nbins=50,
	title='EDA 4: Basket Size Distribution (Unique Items per Invoice)'
)
fig4.add_vline(x=2, line_dash='dash', line_color='#E63946', annotation_text='Min. Market Basket (2 Items)')
fig4.update_layout(xaxis_title='Number of Product Types within 1 Receipt', yaxis_title='Number of Transactions')
fig4.show()

In [26]:
# ==========================================
# CHART 5: TOP 10 BEST-SELLING PRODUCTS VS TOP 10 RETURNED PRODUCTS
# ==========================================
top_sold = (
	clean_df.groupby('Description', as_index=False)['Quantity']
	.sum().sort_values('Quantity', ascending=True).tail(10)
)
top_ret = (
	returns_df.assign(ReturnedQty=returns_df['Quantity'].abs())
	.groupby('Description', as_index=False)['ReturnedQty']
	.sum().sort_values('ReturnedQty', ascending=True).tail(10)
)

fig5 = make_subplots(rows=2, cols=1, subplot_titles=('Top 10 Best-Selling Products (Units)', 'Top 10 Returned Products (Units)'))
fig5.add_trace(go.Bar(x=top_sold['Quantity'], y=top_sold['Description'], orientation='h', marker_color='#2E86AB', name='Sold'), row=1, col=1)
fig5.add_trace(go.Bar(x=top_ret['ReturnedQty'], y=top_ret['Description'], orientation='h', marker_color='#E63946', name='Returned'), row=2, col=1)
fig5.update_layout(width=1000,title_text='EDA 5: Comparison of Volume-Driven Products vs. Return Risk', showlegend=False, height=800)
fig5.show()

In [27]:
# ==========================================
# CHART 6: TOP 10 NON-UK COUNTRIES (CHANCES TO EXPANSION)
# ==========================================
non_uk = (
	clean_df[clean_df['Country'] != 'United Kingdom']
	.groupby('Country', as_index=False)['TotalRevenue']
	.sum().sort_values('TotalRevenue', ascending=True).tail(10)
)
fig6 = px.bar(
	non_uk, x='TotalRevenue', y='Country', orientation='h',
	text_auto='.3s', color='TotalRevenue', color_continuous_scale='Blues',
	title='EDA 6: Top 10 Non-UK International Market (Total Revenue)'
)
fig6.update_layout(coloraxis_showscale=False, xaxis_title='Total Revenue (£)', yaxis_title='Country')
fig6.show()

### Key Takeaways from Exploratory Data Analysis
* **Strong Q4 Seasonality & Holiday Shutdowns:** Daily demand surges consistently in Q4 ahead of the year-end holiday season, followed by a complete operational halt between late December and early January.
* **B2B / Wholesale Weekly Rhythm:** Zero sales occur on Saturdays and minimal volume on Sundays, while peak order volume concentrates between Monday and Thursday—justifying `day_of_week` and `is_saturday` as critical model features.
* **7-Day Autocorrelation Cycle:** Significant autocorrelation spikes at lags 7, 14, 21, and 28 confirm strong weekly periodicity, validating the use of 7-day lag and rolling window features.
* **Multi-Item Basket Structure:** Most invoices contain between 2 and 30 unique SKUs, confirming strong suitability for association rule mining.

## Data Modeling

In [59]:
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from statsmodels.tsa.statespace.sarimax import SARIMAX

# =======================================================================
# EVALUATION METRICS FUNCTION 
# =======================================================================
def wmape(y_true, y_pred):
  return (np.sum(np.abs(y_true - y_pred)) / np.sum(y_true)) * 100

def compute_metrics(y_true, y_pred):
  y_pred_clean = np.maximum(0, y_pred)
  return {
		'MAE': round(mean_absolute_error(y_true, y_pred_clean), 2),
		'RMSE': round(root_mean_squared_error(y_true, y_pred_clean), 2),
		'wMAPE (%)': round(wmape(y_true, y_pred_clean), 2),
		'R2_Score': round(r2_score(y_true, y_pred_clean),4)
	}
# =======================================================================
# TRAINING PIPELINE, TIME-SERIES CV & EVALUATION (REGRESSION + SARIMAX)
# =======================================================================
def demand_modeling(df_model, naive_col, seasonal_period, test_ratio=0.2):
  exclude_cols = ['Date', 'total_quantity', 'total_revenue', 'unique_invoices']
  feature_cols = [c for c in df_model.columns if c not in exclude_cols]
  
  X = df_model[feature_cols]
  y = df_model['total_quantity']
  dates = df_model['Date']
  
  split_idx = int(len(df_model) * (1 - test_ratio))
  X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
  y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
  dates_test = dates.iloc[split_idx:]
  
  models = {
		'Ridge Regression': Pipeline([
			('scaler', StandardScaler()),
			('model', Ridge(alpha=10.0))
		]),
		'Random Forest': RandomForestRegressor(
        n_estimators=200, max_depth=6, min_samples_leaf=3, random_state=42
    ),
		'Gradient Boosting': GradientBoostingRegressor(
				n_estimators=500, max_depth=7, learning_rate=0.02, random_state=42
		)
	}
  
  # A. TimeSeriesSplit Cross-Validation on Train Set (5 Folds)
  tscv = TimeSeriesSplit(n_splits=5)
  cv_summary = {}
  for name, model in models.items():
    fold_rmse = []
    for tr_idx, val_idx in tscv.split(X_train):
      model.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])      
      pred_val = np.maximum(0, model.predict(X_train.iloc[val_idx]))
      fold_rmse.append(root_mean_squared_error(y_train.iloc[val_idx], pred_val))
    cv_summary[name] = round(np.mean(fold_rmse), 2)
  
  # B. Evaluation on the Holdout Test Set
  results = []
  predictions = {}
  
  # 1. Seasonal Naive Baseline
  naive_pred = X_test[naive_col].values
  predictions['Seasonal Naive'] = naive_pred
  results.append({'Model': 'Seasonal Naive (Baseline)', 'CV_RMSE': '-', **compute_metrics(y_test, naive_pred)})
  
  # 2. Tabular Regression Models
  trained_models = {}
  for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = np.maximum(0, model.predict(X_test))
    predictions[name] = y_pred
    trained_models[name] = model
    results.append({'Model': name, 'CV_RMSE': cv_summary[name], **compute_metrics(y_test, y_pred)})
  
  # 3. Statistical Time Series (SARIMAX)
  sarimax_model = SARIMAX(
		y_train,
		order=(1, 0, 1),
		seasonal_order=(1, 1, 1, seasonal_period),
		enforce_stationarity=False,
		enforce_invertibility=False
	).fit(disp=False)
  
  sarimax_pred = np.maximum(0, sarimax_model.forecast(steps=len(y_test)).values)
  predictions['SARIMAX'] = sarimax_pred
  results.append({'Model': f'SARIMAX (s={seasonal_period})', 'CV_RMSE': '-', **compute_metrics(y_test, sarimax_pred)})
  
  metrics_df = pd.DataFrame(results).sort_values('RMSE').reset_index(drop=True)
  return metrics_df, dates_test, y_test, predictions, trained_models, feature_cols 

### Evaluation

In [60]:
daily_metrics, d_dates, d_y_test, d_preds, d_models, d_features = demand_modeling(
	daily_model_df, naive_col='qty_lag_7', seasonal_period=7
)
print("=== Daily Pred Metrics ===")
print(daily_metrics.to_string(index=False))

=== Daily Pred Metrics ===
                    Model   CV_RMSE     MAE     RMSE  wMAPE (%)  R2_Score
            Random Forest   9155.92 5824.65  9576.73      29.34    0.5051
         Ridge Regression   9532.62 6510.56  9868.37      32.79    0.4745
Seasonal Naive (Baseline)         - 6403.95 10091.25      32.26    0.4505
        Gradient Boosting  10640.21 6416.31 11204.06      32.32    0.3226
            SARIMAX (s=7)         - 8484.69 12804.98      42.74    0.1152


In [61]:
weekly_metrics, w_dates, w_y_test, w_preds, w_models, w_features = demand_modeling(
	weekly_model_df, naive_col='qty_lag_1', seasonal_period=4
)
print("=== Weekly Pred Metrics ===")
print(weekly_metrics.to_string(index=False))

=== Weekly Pred Metrics ===
                    Model   CV_RMSE      MAE     RMSE  wMAPE (%)  R2_Score
Seasonal Naive (Baseline)         - 23724.14 33110.12      17.28    0.2811
         Ridge Regression  38112.01 27397.46 36126.75      19.96    0.1441
            Random Forest  35462.19 30198.11 40534.86      22.00   -0.0775
        Gradient Boosting  44643.31 36912.07 45958.75      26.89   -0.3851
            SARIMAX (s=4)         - 58915.82 71700.55      42.91   -2.3713


### Visualization

In [31]:
# =========================================================================
# VISUALIZATION 1: ACTUAL VS PREDICTED (DAILY & WEEKLY)
# =========================================================================
fig = make_subplots(
	rows=2, cols=1,
	subplot_titles=(
		'Daily Demand Prediction (Test Set): Actual vs Best Models',
		'Weekly Demand Prediction (Test Set): Actual vs Best Models'
	),
	vertical_spacing=0.15
)

# Daily Plot
fig.add_trace(go.Scatter(x=d_dates, y=d_y_test, name='Actual (Daily)', line=dict(color='#1D3557', width=2), mode='lines+markers'), row=1, col=1)
fig.add_trace(go.Scatter(x=d_dates, y=d_preds['Random Forest'], name='Random Forest (Daily)', line=dict(color='#E63946', dash='dash')), row=1, col=1)
fig.add_trace(go.Scatter(x=d_dates, y=d_preds['SARIMAX'], name='SARIMAX (Daily)', line=dict(color='#2A9D8F', dash='dot')), row=1, col=1)

# Weekly Plot
fig.add_trace(go.Scatter(x=w_dates, y=w_y_test, name='Actual (Weekly)', line=dict(color='#1D3557', width=2), mode='lines+markers'), row=2, col=1)
fig.add_trace(go.Scatter(x=w_dates, y=w_preds['Ridge Regression'], name='Ridge Regression (Weekly)', line=dict(color='#E63946', dash='dash')), row=2, col=1)
fig.add_trace(go.Scatter(x=w_dates, y=w_preds['Random Forest'], name='Random Forest (Weekly)', line=dict(color='#F4A261', dash='dot')), row=2, col=1)

fig.update_layout(height=700, title_text='Visual Evaluation of Forecasting Performance During the Test Period', hovermode='x unified')
fig.show()

In [32]:
# =========================================================================
# VISUALIZATION 2: FEATURE IMPORTANCE (DEMAND DRIVERS)
# =========================================================================
rf_daily = d_models['Random Forest']
imp_df = pd.DataFrame({
	'Feature': d_features,
	'Importance': rf_daily.feature_importances_
}).sort_values('Importance', ascending=True).tail(10)

fig_imp = go.Figure(go.Bar(
	x=imp_df['Importance'], y=imp_df['Feature'],
	orientation='h', marker_color='#457B9D'
))
fig_imp.update_layout(
	title='Top 10 Factors Driving Daily Demand (Feature Importance)',
	xaxis_title='Relative Importance', yaxis_title='Feature'
)
fig_imp.show()

### Demand Modeling & Evaluation Insights
* **Daily Forecasting Winner — Random Forest (`RMSE: 9,576.73` | `wMAPE: 29.34%` | `R²: 0.5051`):** Random Forest outperforms the Seasonal Naive baseline across all metrics. Its test RMSE closely aligns with its 5-fold Time-Series CV RMSE (`9,155.92`), demonstrating strong generalization without overfitting.
* **Weekly Forecasting Dynamics — Baseline Wins (`RMSE: 33,110.12` | `wMAPE: 17.28%`):** At the weekly level, `Seasonal Naive (qty_lag_1)` outperforms complex ML models (`Ridge: 19.96% wMAPE`, `Random Forest: 22.00% wMAPE`). With only ~80 training weeks available and the test split falling directly on the Q4 peak surge, tree-based models struggle to extrapolate beyond historical training ranges, whereas a 1-week lag immediately adapts to momentum.
* **Operational Takeaway:** Weekly aggregation cuts relative forecast error (`wMAPE`) nearly in half compared to daily predictions. **Random Forest** should be used for daily warehouse staffing and dispatch scheduling, while **Naive Lag-1 / Ridge Regression** is optimal for weekly inventory replenishment.

## Market Basket Analysis

### Basket Preparation & FP-Growth Modeling

In [33]:
from mlxtend.frequent_patterns import fpgrowth, association_rules

# 1. Description Standardization per StockCode
canonical_desc = (
	clean_df.groupby('StockCode')['Description']
	.agg(lambda x: x.mode().iloc[0])
)
mba_df = clean_df.copy()
mba_df['CleanDescription'] = mba_df['StockCode'].map(canonical_desc)
mba_df

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,TotalRevenue,Date,CleanDescription
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.40,2009-12-01,15CM CHRISTMAS GLASS BALL 20 LIGHTS
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00,2009-12-01,PINK CHERRY LIGHTS
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.00,2009-12-01,WHITE CHERRY LIGHTS
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.80,2009-12-01,"RECORD FRAME 7"" SINGLE SIZE"
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.00,2009-12-01,STRAWBERRY CERAMIC TRINKET BOX
...,...,...,...,...,...,...,...,...,...,...,...
1067365,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680,France,10.20,2011-12-09,PACK OF 20 SPACEBOY NAPKINS
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680,France,12.60,2011-12-09,CHILDREN'S APRON DOLLY GIRL
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680,France,16.60,2011-12-09,CHILDRENS CUTLERY DOLLY GIRL
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680,France,16.60,2011-12-09,CHILDRENS CUTLERY CIRCUS PARADE


In [35]:
# 2. Filter UK Transaction (main market, ~90% transactions) & invoice with at least 2 unique items
uk_df = mba_df[mba_df['Country'] == 'United Kingdom']
invoice_item_counts = uk_df.groupby('Invoice')['StockCode'].nunique()
valid_invoices = invoice_item_counts[invoice_item_counts >= 2].index

filtered_basket = uk_df[uk_df['Invoice'].isin(valid_invoices)]
filtered_basket

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country,TotalRevenue,Date,CleanDescription
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4,2009-12-01,15CM CHRISTMAS GLASS BALL 20 LIGHTS
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0,2009-12-01,PINK CHERRY LIGHTS
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0,2009-12-01,WHITE CHERRY LIGHTS
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8,2009-12-01,"RECORD FRAME 7"" SINGLE SIZE"
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0,2009-12-01,STRAWBERRY CERAMIC TRINKET BOX
...,...,...,...,...,...,...,...,...,...,...,...
1067350,581585,22466,FAIRY TALE COTTAGE NIGHT LIGHT,12,2011-12-09 12:31:00,1.95,15804,United Kingdom,23.4,2011-12-09,FAIRY TALE COTTAGE NIGHTLIGHT
1067351,581586,22061,LARGE CAKE STAND HANGING STRAWBERY,8,2011-12-09 12:49:00,2.95,13113,United Kingdom,23.6,2011-12-09,LARGE CAKE STAND HANGING STRAWBERY
1067352,581586,23275,SET OF 3 HANGING OWLS OLLIE BEAK,24,2011-12-09 12:49:00,1.25,13113,United Kingdom,30.0,2011-12-09,SET OF 3 HANGING OWLS OLLIE BEAK
1067353,581586,21217,RED RETROSPOT ROUND CAKE TINS,24,2011-12-09 12:49:00,8.95,13113,United Kingdom,214.8,2011-12-09,RED RETROSPOT ROUND CAKE TINS


In [36]:
# 3. One-Hot Encoded matrix (Boolean) per Invoice x Product
basket_matrix = (
	filtered_basket.groupby(['Invoice', 'CleanDescription'])['Quantity'].sum()
	.unstack(fill_value=0)
)
basket_bool = basket_matrix > 0

print(f"Valid Basket Total (UK, >=2 items): {basket_bool.shape[0]:,} transactions")
print(f"Unique Product Total: {basket_bool.shape[1]:,} item")

Valid Basket Total (UK, >=2 items): 33,163 transactions
Unique Product Total: 4,691 item


In [38]:
# 4. Frequent Itemset Extraction with FP-Growth (min_support = 2%)
frequent_itemsets = fpgrowth(basket_bool, min_support=0.02, use_colnames=True)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)
frequent_itemsets

,support,itemsets,length
0,0.067967,frozenset({STRAWBERRY CERAMIC TRINKET BOX}),1
1,0.031360,frozenset({DOORMAT FANCY FONT HOME SWEET HOME}),1
2,0.021681,frozenset({SAVE THE PLANET MUG}),1
3,0.022646,frozenset({LARGE HEART MEASURING SPOONS}),1
4,0.079486,frozenset({ASSORTED COLOUR BIRD ORNAMENT}),1
...,...,...,...
406,0.023701,"frozenset({PINK REGENCY TEACUP AND SAUCER, GRE...",2
407,0.022284,"frozenset({PINK REGENCY TEACUP AND SAUCER, ROS...",2
408,0.020143,"frozenset({PINK REGENCY TEACUP AND SAUCER, GRE...",3
409,0.024757,"frozenset({PAPER CHAIN KIT 50'S CHRISTMAS, PAP...",2


In [39]:

# 5. Generate Association Rules (Lift >= 1.5 & Confidence >= 30%)
rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1.5)
rules = rules[rules['confidence'] >= 0.30].sort_values(['lift', 'confidence'], ascending=False)
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
183,frozenset({PINK REGENCY TEACUP AND SAUCER}),"frozenset({GREEN REGENCY TEACUP AND SAUCER, RO...",0.028405,0.028465,0.020143,0.709130,24.911930,1.0,0.019334,3.340093,0.987921,0.548440,0.700607,0.708378
182,"frozenset({GREEN REGENCY TEACUP AND SAUCER, RO...",frozenset({PINK REGENCY TEACUP AND SAUCER}),0.028465,0.028405,0.020143,0.707627,24.911930,1.0,0.019334,3.323136,0.987982,0.548440,0.699079,0.708378
181,"frozenset({PINK REGENCY TEACUP AND SAUCER, ROS...",frozenset({GREEN REGENCY TEACUP AND SAUCER}),0.022284,0.037451,0.020143,0.903924,24.135941,1.0,0.019308,10.018640,0.980415,0.508759,0.900186,0.720883
184,frozenset({GREEN REGENCY TEACUP AND SAUCER}),"frozenset({PINK REGENCY TEACUP AND SAUCER, ROS...",0.037451,0.022284,0.020143,0.537842,24.135941,1.0,0.019308,2.115546,0.995865,0.508759,0.527309,0.720883
176,frozenset({PINK REGENCY TEACUP AND SAUCER}),frozenset({GREEN REGENCY TEACUP AND SAUCER}),0.028405,0.037451,0.023701,0.834395,22.279419,1.0,0.022637,5.812313,0.983039,0.562232,0.827951,0.733623
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
172,frozenset({HEART OF WICKER LARGE}),frozenset({WHITE HANGING HEART T-LIGHT HOLDER}),0.058016,0.157073,0.022827,0.393451,2.504899,1.0,0.013714,1.389711,0.637784,0.118726,0.280426,0.269388
168,frozenset({HEART OF WICKER SMALL}),frozenset({WHITE HANGING HEART T-LIGHT HOLDER}),0.062962,0.157073,0.023038,0.365900,2.329498,1.0,0.013148,1.329330,0.609071,0.116945,0.247741,0.256285
5,frozenset({HOME BUILDING BLOCK WORD}),frozenset({WHITE HANGING HEART T-LIGHT HOLDER}),0.060188,0.157073,0.022012,0.365731,2.328422,1.0,0.012559,1.328976,0.607062,0.112741,0.247541,0.252937
167,frozenset({NATURAL SLATE HEART CHALKBOARD}),frozenset({WHITE HANGING HEART T-LIGHT HOLDER}),0.059615,0.157073,0.021078,0.353566,2.250971,1.0,0.011714,1.303965,0.590978,0.107754,0.233108,0.243878


In [43]:
# Format frozenset into a well-formatted string
rules['antecedents_str'] = rules['antecedents'].apply(lambda x: ', '.join(list(x)))
rules['consequents_str'] = rules['consequents'].apply(lambda x: ', '.join(list(x)))
rules['rule_label'] = rules['antecedents_str'] + ' -> ' + rules['consequents_str']
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski,antecendents_str,consequences_str,antecedents_str,consequents_str,rule_label
183,frozenset({PINK REGENCY TEACUP AND SAUCER}),"frozenset({GREEN REGENCY TEACUP AND SAUCER, RO...",0.028405,0.028465,0.020143,0.709130,24.911930,1.0,0.019334,3.340093,0.987921,0.548440,0.700607,0.708378,PINK REGENCY TEACUP AND SAUCER,"GREEN REGENCY TEACUP AND SAUCER, ROSES REGENCY...",PINK REGENCY TEACUP AND SAUCER,"GREEN REGENCY TEACUP AND SAUCER, ROSES REGENCY...",PINK REGENCY TEACUP AND SAUCER -> GREEN REGENC...
182,"frozenset({GREEN REGENCY TEACUP AND SAUCER, RO...",frozenset({PINK REGENCY TEACUP AND SAUCER}),0.028465,0.028405,0.020143,0.707627,24.911930,1.0,0.019334,3.323136,0.987982,0.548440,0.699079,0.708378,"GREEN REGENCY TEACUP AND SAUCER, ROSES REGENCY...",PINK REGENCY TEACUP AND SAUCER,"GREEN REGENCY TEACUP AND SAUCER, ROSES REGENCY...",PINK REGENCY TEACUP AND SAUCER,"GREEN REGENCY TEACUP AND SAUCER, ROSES REGENCY..."
181,"frozenset({PINK REGENCY TEACUP AND SAUCER, ROS...",frozenset({GREEN REGENCY TEACUP AND SAUCER}),0.022284,0.037451,0.020143,0.903924,24.135941,1.0,0.019308,10.018640,0.980415,0.508759,0.900186,0.720883,"PINK REGENCY TEACUP AND SAUCER, ROSES REGENCY ...",GREEN REGENCY TEACUP AND SAUCER,"PINK REGENCY TEACUP AND SAUCER, ROSES REGENCY ...",GREEN REGENCY TEACUP AND SAUCER,"PINK REGENCY TEACUP AND SAUCER, ROSES REGENCY ..."
184,frozenset({GREEN REGENCY TEACUP AND SAUCER}),"frozenset({PINK REGENCY TEACUP AND SAUCER, ROS...",0.037451,0.022284,0.020143,0.537842,24.135941,1.0,0.019308,2.115546,0.995865,0.508759,0.527309,0.720883,GREEN REGENCY TEACUP AND SAUCER,"PINK REGENCY TEACUP AND SAUCER, ROSES REGENCY ...",GREEN REGENCY TEACUP AND SAUCER,"PINK REGENCY TEACUP AND SAUCER, ROSES REGENCY ...",GREEN REGENCY TEACUP AND SAUCER -> PINK REGENC...
176,frozenset({PINK REGENCY TEACUP AND SAUCER}),frozenset({GREEN REGENCY TEACUP AND SAUCER}),0.028405,0.037451,0.023701,0.834395,22.279419,1.0,0.022637,5.812313,0.983039,0.562232,0.827951,0.733623,PINK REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,PINK REGENCY TEACUP AND SAUCER,GREEN REGENCY TEACUP AND SAUCER,PINK REGENCY TEACUP AND SAUCER -> GREEN REGENC...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
172,frozenset({HEART OF WICKER LARGE}),frozenset({WHITE HANGING HEART T-LIGHT HOLDER}),0.058016,0.157073,0.022827,0.393451,2.504899,1.0,0.013714,1.389711,0.637784,0.118726,0.280426,0.269388,HEART OF WICKER LARGE,WHITE HANGING HEART T-LIGHT HOLDER,HEART OF WICKER LARGE,WHITE HANGING HEART T-LIGHT HOLDER,HEART OF WICKER LARGE -> WHITE HANGING HEART T...
168,frozenset({HEART OF WICKER SMALL}),frozenset({WHITE HANGING HEART T-LIGHT HOLDER}),0.062962,0.157073,0.023038,0.365900,2.329498,1.0,0.013148,1.329330,0.609071,0.116945,0.247741,0.256285,HEART OF WICKER SMALL,WHITE HANGING HEART T-LIGHT HOLDER,HEART OF WICKER SMALL,WHITE HANGING HEART T-LIGHT HOLDER,HEART OF WICKER SMALL -> WHITE HANGING HEART T...
5,frozenset({HOME BUILDING BLOCK WORD}),frozenset({WHITE HANGING HEART T-LIGHT HOLDER}),0.060188,0.157073,0.022012,0.365731,2.328422,1.0,0.012559,1.328976,0.607062,0.112741,0.247541,0.252937,HOME BUILDING BLOCK WORD,WHITE HANGING HEART T-LIGHT HOLDER,HOME BUILDING BLOCK WORD,WHITE HANGING HEART T-LIGHT HOLDER,HOME BUILDING BLOCK WORD -> WHITE HANGING HEAR...
167,frozenset({NATURAL SLATE HEART CHALKBOARD}),frozenset({WHITE HANGING HEART T-LIGHT HOLDER}),0.059615,0.157073,0.021078,0.353566,2.250971,1.0,0.011714,1.303965,0.590978,0.107754,0.233108,0.243878,NATURAL SLATE HEART CHALKBOARD,WHITE HANGING HEART T-LIGHT HOLDER,NATURAL SLATE HEART CHALKBOARD,WHITE HANGING HEART T-LIGHT HOLDER,NATURAL SLATE HEART CHALKBOARD -> WHITE HANGIN...


In [47]:
print(f"\nTotal Association Rules Found: {len(rules)}")
print("\n=== TOP 15 PRODUCT BUNDLING RULES (BY LIFT) ===")
cols_show = ['antecedents_str', 'consequents_str', 'support', 'confidence', 'lift']
print(rules[cols_show].head(15).round(4).to_string(index=False))


Total Association Rules Found: 152

=== TOP 15 PRODUCT BUNDLING RULES (BY LIFT) ===
                                                 antecedents_str                                                  consequents_str  support  confidence    lift
                                  PINK REGENCY TEACUP AND SAUCER GREEN REGENCY TEACUP AND SAUCER, ROSES REGENCY TEACUP AND SAUCER   0.0201      0.7091 24.9119
GREEN REGENCY TEACUP AND SAUCER, ROSES REGENCY TEACUP AND SAUCER                                   PINK REGENCY TEACUP AND SAUCER   0.0201      0.7076 24.9119
 PINK REGENCY TEACUP AND SAUCER, ROSES REGENCY TEACUP AND SAUCER                                  GREEN REGENCY TEACUP AND SAUCER   0.0201      0.9039 24.1359
                                 GREEN REGENCY TEACUP AND SAUCER  PINK REGENCY TEACUP AND SAUCER, ROSES REGENCY TEACUP AND SAUCER   0.0201      0.5378 24.1359
                                  PINK REGENCY TEACUP AND SAUCER                                  GREEN REGENCY TEACUP A

### Association Rules Visualization

In [58]:
# =========================================================================
# VISUALIZATION: MARKET BASKET ASSOCIATION RULES & TOP BUNDLING PAIRS
# =========================================================================
top_rules = rules.head(10).sort_values('lift', ascending=True)

fig_mba = make_subplots(
  rows=2, cols=1,
  subplot_titles=(
    'Association Rules Landscape (Support vs. Confidence, Colored by Lift)',
    'Top 10 Strongest Product Bundling Pairs (Ranked by Lift)'
  ),
  vertical_spacing=0.15
)

# Panel 1: Scatter Plot Support vs Confidence
fig_mba.add_trace(
  go.Scatter(
    x=rules['support'] * 100,
    y=rules['confidence'] * 100,
    mode='markers',
    text=rules['rule_label'],
    marker=dict(
      size=np.clip(rules['lift'] * 3.5, 8, 28),
      color=rules['lift'],
      colorscale='Tealrose',
      showscale=True,
      colorbar=dict(title='Lift', y=0.78, len=0.4),
      opacity=0.85
    ),
    hovertemplate=(
      '<b>%{text}</b><br>'
      'Support: %{x:.2f}%<br>'
      'Confidence: %{y:.2f}%<br>'
      'Lift: %{marker.color:.2f}x<extra></extra>'
    )
  ),
  row=1, col=1
)

# Panel 2: Horizontal Bar Top 10 Rules by Lift
fig_mba.add_trace(
  go.Bar(
    x=top_rules['lift'],
    y=top_rules['rule_label'],
    orientation='h',
    marker_color='#2E86AB',
    text=top_rules.apply(lambda r: f"Lift: {r['lift']:.2f}x | Conf: {r['confidence']*100:.0f}%", axis=1),
    textposition='auto'
  ),
  row=2, col=1
)

fig_mba.update_xaxes(title_text='Support (% of Transactions)', row=1, col=1)
fig_mba.update_yaxes(title_text='Confidence (%)', row=1, col=1)
fig_mba.update_xaxes(title_text='Lift Ratio (Strength of Association)', row=2, col=1)

fig_mba.update_layout(
  height=900,
  width=1300,
  title_text='Market Basket Analysis: Product Co-Occurrence & Cross-Selling Opportunities',
  showlegend=False
)
fig_mba.show()

### Market Basket Analysis Insights
* **High-Affinity Product Sets:** The *Regency Teacup and Saucer* product line (Pink, Green, and Roses variants) dominates the top association rules, achieving **Lift ratios between 20.4x and 24.9x**.
* **Strong Cross-Sell Conversion:** Customers who add both the *Pink* and *Roses* variants to their cart have a **90% confidence** of also purchasing the *Green* variant.
* **Merchandising Action:** Rather than discounting individual bestsellers, these items should be packaged as a *"Regency Tea Set Trio"* bundle and surfaced via automated *"Frequently Bought Together"* recommendations at checkout to increase Average Order Value (AOV).

### Risk Management Analytics & Visualization

In [ ]:
# =============================================================================
# 1. PRODUCT RETURN RISK MATRIX (SALES VOLUME VS RETURN RATE)
# =============================================================================
sold_summary = clean_df.groupby(['StockCode', 'Description'], as_index=False).agg(
	Sold_Qty=('Quantity','sum'),
	Gross_Revenue=('TotalRevenue','sum'),
	Avg_Price=('Price','mean')
)

ret_summary = (
	returns_df.assign(
		Ret_Qty=returns_df['Quantity'].abs(),
		Ret_Loss=(returns_df['Quantity'].abs() * returns_df['Price'])
  )
	.groupby('StockCode', as_index=False)
	.agg(Returned_Qty=('Ret_Qty','sum'), Return_Loss=('Ret_Loss','sum'))
)

prod_risk = pd.merge(sold_summary, ret_summary, on='StockCode', how='left').fillna(0)
# Focus on active products with sales volume >= 500 units 
prod_risk = prod_risk[prod_risk['Sold_Qty'] >= 500].copy()
prod_risk['Return_Rate (%)'] = (prod_risk['Returned_Qty'] / prod_risk['Sold_Qty']) * 100

high_return_products = prod_risk.sort_values('Return_Loss', ascending=False).head(20)
print("=== TOP 20 PRODUCTS WITH THE HIGHEST RISK OF RETURN LOSSES ===")
print(high_return_products[['StockCode', 'Description', 'Sold_Qty', 'Returned_Qty', 'Return_Rate (%)', 'Return_Loss']].to_string(index=False))

=== TOP 20 PRODUCTS WITH THE HIGHEST RISK OF RETURN LOSSES ===
StockCode                        Description  Sold_Qty  Returned_Qty  Return_Rate (%)  Return_Loss
    23843        PAPER CRAFT , LITTLE BIRDIE     80995       80995.0       100.000000    168469.60
    23166     MEDIUM CERAMIC TOP STORAGE JAR     78033       74494.0        95.464739     77479.64
    22423           REGENCY CAKESTAND 3 TIER     26478        1799.0         6.794320     16545.30
   85123A WHITE HANGING HEART T-LIGHT HOLDER     94142        4085.0         4.339190      9387.10
    71477  COLOUR GLASS. STAR T-LIGHT HOLDER     14077        3240.0        23.016268      7164.12
    71477 COLOURED GLASS STAR T-LIGHT HOLDER      1012        3240.0       320.158103      7164.12
    21108 FAIRY CAKE FLANNEL ASSORTED COLOUR     12962        3159.0        24.371239      6614.37
   79323W                WHITE CHERRY LIGHTS      3004        6054.0       201.531292      6421.10
    21843           RED RETROSPOT CAKE STAND  

In [ ]:
# =============================================================================
# 2. CUSTOMER RISK SEGMENTATION (RFM & REVENUE CONCENTRATION RISK)
# =============================================================================
reg_customers = clean_df[clean_df['CustomerID'] != 'Guest'].copy()
snapshot_date = reg_customers['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = reg_customers.groupby('CustomerID', as_index=False).agg(
	Recency=('InvoiceDate',lambda x: (snapshot_date - x.max()).days),
	Frequency=('Invoice','nunique'),
	Monetary=('TotalRevenue','sum')
)

rfm['R_Score'] = pd.qcut(rfm['Recency'], 4, labels=[4,3,2,1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 4, labels=[1,2,3,4]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 4, labels=[1,2,3,4]).astype(int)

def classify_risk_segment(row):
  if row['R_Score'] >= 3 and row['F_Score'] >= 3 and row['M_Score'] >= 3:
    return 'Champions (Low Risk)'
  elif row['R_Score'] <= 2 and row['M_Score'] >= 3:
    return 'At-Risk High Value (Critical Churn Risk)'
  elif row['R_Score'] <= 2 and row['F_Score'] <= 2:
    return 'Hibernating / Lost (High Churn)'
  elif row['R_Score'] >=3 and row['F_Score'] <= 2:
    return 'Recent / Promising (Onboarding Risk)'
  else:
    return 'Loyal / Potential (Moderate Risk)'
  
rfm['Risk_Segment'] = rfm.apply(classify_risk_segment, axis=1)

segment_summary = rfm.groupby('Risk_Segment', as_index=False).agg(
	Customer_Count=('CustomerID','count'),
	Total_Revenue=('Monetary','sum'),
	Avg_Recency_Days=('Recency','mean'),
	Avg_Frequency=('Frequency','mean')
).sort_values('Total_Revenue', ascending=False)

segment_summary['Revenue_Share (%)'] = (segment_summary['Total_Revenue'] / segment_summary['Total_Revenue'].sum()) * 100

print("\n=== CUSTOMER RISK SEGMENTS SUMMARY ===")
print(segment_summary.to_string(index=False))


=== CUSTOMER RISK SEGMENTS SUMMARY ===
                            Risk_Segment  Customer_Count  Total_Revenue  Avg_Recency_Days  Avg_Frequency  Revenue_Share (%)
                    Champions (Low Risk)            1816   12983818.757         29.357930      13.980176          76.048493
At-Risk High Value (Critical Churn Risk)             911    2506722.557        295.488474       5.781559          14.682312
    Recent / Promising (Onboarding Risk)             895     749175.731         40.436872       1.770950           4.388053
         Hibernating / Lost (High Churn)            1764     554669.401        418.830499       1.384921           3.248796
       Loyal / Potential (Moderate Risk)             475     278691.861        168.810526       4.117895           1.632347


In [ ]:
# =============================================================================
# 2. CUSTOMER RISK SEGMENTATION (RFM & REVENUE CONCENTRATION RISK)
# =============================================================================
fig_risk = make_subplots(
	rows=2, cols=1,
	subplot_titles=(
		'Product Return Risk Matrix (Sold Volume vs Return Rate %)',
		'Revenue Exposure by Customer Risk Segment'
	)
)

# Left Panel: Scatter Risk Matrix Product
fig_risk.add_trace(go.Scatter(
	x=prod_risk['Sold_Qty'], y=prod_risk['Return_Rate (%)'], mode='markers', text=prod_risk['Description'],
	marker=dict(size=np.clip(prod_risk['Return_Loss'] / 200, 6, 35), color=prod_risk['Return_Rate (%)'], colorscale='plasma', showscale=False, opacity=1),
	hovertemplate='<b>%{text}</b><br>Sold: %{x:,}<br>Return Rate: %{y:.2f}%<extra></extra>'
), row=1, col=1)

# Right Panel: Bar Chart Revenue Exposure by Risk Segment
fig_risk.add_trace(go.Bar(
	x=segment_summary['Total_Revenue'], y=segment_summary['Risk_Segment'], orientation='h',
	marker_color=['#E63946' if 'Critical' in s else '#457B9D' for s in segment_summary['Risk_Segment']],
	text=segment_summary['Revenue_Share (%)'].map(lambda v: f'{v:.1f}%'),
	textposition='auto'
), row=2, col=1)

fig_risk.update_xaxes(type='log', title_text='Total Units Sold (Log Scale)', row=1, col=1)
fig_risk.update_yaxes(title_text='Return Rate (%)', row=1, col=1)
fig_risk.update_xaxes(title_text='Total Revenue (£)', row=2, col=1)
fig_risk.update_layout(height=1000, width=1000, title_text='Retail Risk Management Dashboard: Product Return & Customer Attrition Exposure', showlegend=False)
fig_risk.show()

## Executive Summary & Strategic Recommendations

By combining demand forecasting, basket analysis, and commercial risk profiling, this analysis translates transactional logs into four actionable business initiatives:

1. **Tiered Forecasting Deployment:** Deploy the **Random Forest** model for daily operational scheduling (accounting for Saturday closures and weekly lag cycles) and use **Weekly Lag-1 / Ridge** benchmarks for macro procurement planning during Q4 surges.
2. **Targeted Product Bundling:** Launch tiered bundle promotions around high-lift product families (*Regency Teacup & Saucer* series, Lift > 20x) to capture high-confidence cross-sell opportunities.
3. **Return Loss Mitigation:** Audit top revenue-draining return items identified in the *Product Return Risk Matrix* to address recurring bulk-order cancellations, packaging damage, or listing inaccuracies.
4. **Proactive Retention for At-Risk VIPs:** Prioritize account-manager outreach and personalized re-engagement incentives for the **At-Risk High Value** RFM segment to protect core wholesale revenue concentration.